---
# Assignment Code: DS-AG-031
---

# Q.No. 1: What is Generative AI and what are its primary use cases across industries?

Answer:

Generative AI is a type of Artificial Intelligence that creates new content such as text, images, music, videos, and computer code by learning patterns from existing data. Unlike traditional AI, which mainly analyzes or classifies data, Generative AI generates original content based on the input provided by users.

Generative AI is widely used across many industries. In healthcare, it helps generate medical reports and supports drug discovery. In education, it creates study materials, quizzes, and personalized learning content. In finance, it assists in report generation, fraud detection, and customer support. In software development, it helps developers write code, debug programs, and generate documentation. In marketing, it creates advertisements, social media posts, and product descriptions. It is also used in the entertainment industry to generate stories, music, artwork, and videos.

Although Generative AI offers many benefits such as improved productivity, creativity, and automation, it also has challenges like biased outputs, incorrect information, and privacy concerns. Therefore, it should always be used responsibly with proper human supervision.

---

# Q.No. 2: Explain the role of probabilistic modeling in generative models. How do these models differ from discriminative models?

Answer:

Probabilistic modeling helps generative models learn the probability distribution of the training data. Instead of simply memorizing examples, the model learns patterns and relationships in the data so that it can generate new content similar to the original dataset. This allows generative models to create realistic text, images, audio, and other types of data.

Generative models and discriminative models have different purposes. A generative model learns how the data is created and can generate new samples. In contrast, a discriminative model focuses only on distinguishing between different classes and making predictions.

Some key differences are:

Generative models learn the data distribution, while discriminative models learn the decision boundary.
Generative models can create new data, whereas discriminative models cannot.
Generative models are commonly used for text generation, image generation, and machine translation, while discriminative models are mainly used for classification tasks.

---

# Q.No. 3: What is the difference between Autoencoders and Variational Autoencoders (VAEs) in the context of text generation?

Answer:

Autoencoders (AEs) and Variational Autoencoders (VAEs) are neural network models used to learn meaningful representations of data. Both consist of an encoder, which converts the input into a compressed form, and a decoder, which reconstructs the original input.

The main difference is that a standard Autoencoder learns a fixed latent representation of the input, while a Variational Autoencoder learns a probability distribution of the latent space. Instead of encoding each input as a single point, a VAE represents it using a mean and variance, allowing it to generate new and diverse data samples.

In text generation, Autoencoders are mainly used for reconstructing existing sentences, whereas VAEs can generate new sentences by sampling from the learned latent space. This makes VAEs more suitable for creative text generation and language modeling.

Overall, Autoencoders are best for data compression and reconstruction, while VAEs are more powerful for generating new and meaningful text with greater diversity.


---

# Q.No. 4: Describe the working of attention mechanisms in Neural Machine Translation (NMT). Why are they critical?

Answer:

The attention mechanism is an important part of Neural Machine Translation (NMT). It allows the model to focus on the most relevant words in the input sentence while translating each word of the output sentence.

In a traditional encoder-decoder model, the entire input sentence is compressed into a single vector, which can lead to information loss, especially for long sentences. The attention mechanism solves this problem by giving the decoder access to all encoder outputs and assigning different importance (attention weights) to each input word during translation.

This helps the model understand which source words are most relevant for generating the next target word.

Attention mechanisms are critical because they:

Improve translation accuracy.
Handle long and complex sentences more effectively.
Reduce information loss during encoding.
Capture better relationships between words in different languages.
Form the foundation of modern Transformer models used in advanced language translation systems.

Overall, the attention mechanism makes Neural Machine Translation more accurate, efficient, and capable of producing natural-sounding translations.

---
# Q.No. 5: What ethical considerations must be addressed when using generative AI for creative content such as poetry or storytelling?

Answer:

Generative AI can create poems, stories, and other creative content, but its use also raises several ethical concerns. One important issue is copyright, as AI models may generate content that is similar to existing works, leading to intellectual property problems.

Another concern is bias. If the training data contains biased or unfair information, the AI may produce content that reflects those biases. It is also important to prevent the generation of harmful, offensive, or misleading content.

Transparency is another key consideration. Readers should know whether a piece of creative content was written by a human or generated by AI. In addition, user privacy must be protected by ensuring that personal or sensitive information is not misused during training or content generation.

Finally, human review is essential before publishing AI-generated content to ensure quality, originality, and ethical standards.

In conclusion, responsible use of Generative AI requires fairness, transparency, privacy protection, respect for copyright, and proper human oversight.


---

# Q.No. 6: Use the following small text dataset to train a simple Variational Autoencoder (VAE) for text reconstruction.

In [3]:
# Import Libraries
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import backend as K

# Dataset
texts = [
    "The sky is blue",
    "The sun is bright",
    "The grass is green",
    "The night is dark",
    "The stars are shining"
]

# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)

max_len = max(len(seq) for seq in sequences)
vocab_size = len(tokenizer.word_index) + 1

X = pad_sequences(sequences, maxlen=max_len, padding='post')

# Encoder
latent_dim = 8

inputs = Input(shape=(max_len,))
x = Embedding(vocab_size, 16)(inputs)
x = LSTM(32)(x)

z_mean = Dense(latent_dim)(x)
z_log_var = Dense(latent_dim)(x)

def sampling(args):
    z_mean, z_log_var = args
    epsilon = K.random_normal(shape=(K.shape(z_mean)[0], latent_dim))
    return z_mean + K.exp(0.5 * z_log_var) * epsilon

z = Lambda(sampling)([z_mean, z_log_var])

# Decoder
decoder = Dense(max_len * vocab_size, activation='softmax')(z)
outputs = tf.keras.layers.Reshape((max_len, vocab_size))(decoder)

vae = Model(inputs, outputs)

# VAE Loss
reconstruction_loss = tf.keras.losses.sparse_categorical_crossentropy(X, outputs)
reconstruction_loss = tf.reduce_mean(reconstruction_loss)

kl_loss = -0.5 * tf.reduce_mean(
    1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)
)

vae.add_loss(reconstruction_loss + kl_loss)

vae.compile(optimizer='adam')

# Train Model
vae.fit(X, epochs=100, batch_size=2, verbose=1)

# Reconstruction
pred = vae.predict(X)
predicted_tokens = np.argmax(pred, axis=-1)

print("\nOriginal Sentences:")
for sentence in texts:
    print(sentence)

print("\nReconstructed Sentences:")
for seq in predicted_tokens:
    words = []
    for idx in seq:
        if idx != 0:
            words.append(tokenizer.index_word.get(idx, ""))
    print(" ".join(words))

---
# Q.No. 7: Use a pre-trained GPT model (like GPT-2 or GPT-3) to translate a short English paragraph into French and German.

In [3]:
from transformers import pipeline

translator = pipeline("translation", model="facebook/mbart-large-50-many-to-many-mmt")

text = "Artificial Intelligence is changing the world by making machines smarter and more efficient."

# French Translation
fr = translator(
    text,
    src_lang="en_XX",
    tgt_lang="fr_XX"
)

# German Translation
de = translator(
    text,
    src_lang="en_XX",
    tgt_lang="de_DE"
)

print("Original:")
print(text)

print("\nFrench:")
print(fr[0]["translation_text"])

print("\nGerman:")
print(de[0]["translation_text"])

---
# Q.No. 8: Implement a simple attention-based encoder-decoder model for English-to-Spanish translation using TensorFlow or PyTorch.

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Attention, Concatenate, Dense
from tensorflow.keras.models import Model
import numpy as np

# Dummy Data
encoder_input = np.array([[1, 2, 3, 0],
                          [4, 5, 6, 0]])

decoder_input = np.array([[1, 2, 0],
                          [1, 3, 0]])

decoder_output = np.array([[2, 3, 0],
                           [3, 4, 0]])

vocab_size = 20
embedding_dim = 32
hidden_units = 64

# Encoder
encoder_inputs = Input(shape=(None,))
encoder_embedding = Embedding(vocab_size, embedding_dim)(encoder_inputs)
encoder_outputs, state_h, state_c = LSTM(
    hidden_units,
    return_sequences=True,
    return_state=True
)(encoder_embedding)

# Decoder
decoder_inputs = Input(shape=(None,))
decoder_embedding = Embedding(vocab_size, embedding_dim)(decoder_inputs)
decoder_outputs, _, _ = LSTM(
    hidden_units,
    return_sequences=True,
    return_state=True
)(decoder_embedding, initial_state=[state_h, state_c])

# Attention
attention = Attention()([decoder_outputs, encoder_outputs])

# Concatenate
concat = Concatenate(axis=-1)([decoder_outputs, attention])

# Output Layer
outputs = Dense(vocab_size, activation="softmax")(concat)

# Model
model = Model([encoder_inputs, decoder_inputs], outputs)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Train
model.fit(
    [encoder_input, decoder_input],
    decoder_output,
    epochs=20,
    batch_size=2
)

# Prediction
prediction = model.predict([encoder_input, decoder_input])
print("Prediction Shape:", prediction.shape)

---
# Q.No. 9: Use the following short poetry dataset to simulate poem generation with a pre-trained GPT model.

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model="gpt2")

prompt = """
Roses are red, violets are blue,
Sugar is sweet, and so are you.
The moon glows bright in silent skies,
A bird sings where the soft wind sighs.

Write a new 4-line poem in the same style:
"""

result = generator(
    prompt,
    max_length=90,
    num_return_sequences=1,
    temperature=0.8
)

print("Prompt:")
print(prompt)

print("\nGenerated Poem:")
print(result[0]["generated_text"])

---
# Q.No. 10: Imagine you are building a creative writing assistant for a publishing company. The assistant should generate story plots and character descriptions using Generative AI. Describe how you would design the system, including model selection, training data, bias mitigation, and evaluation methods. Explain the real-world challenges you might face.

Answer:

To build a creative writing assistant, I would use a pre-trained Transformer model such as GPT-2 or GPT-3 because these models are effective at generating natural and creative text. The system would take user inputs such as genre, theme, characters, or writing style and generate story plots, character descriptions, and dialogue.

The training data would include books, novels, movie scripts, short stories, and other high-quality literary content collected from legal and publicly available sources. Before training, the data would be cleaned by removing duplicate, offensive, and irrelevant content to improve the quality of the generated text.

To reduce bias, the training dataset should contain diverse authors, cultures, and writing styles. Content moderation and human review should also be used to prevent the generation of harmful or inappropriate content.

The system can be evaluated using metrics such as BLEU, ROUGE, and perplexity, along with human evaluation for creativity, coherence, grammar, and originality.

Some real-world challenges include copyright issues, biased outputs, misinformation, high computational costs, and maintaining consistency in long stories. Protecting user privacy and ensuring ethical AI usage are also important considerations.

Overall, a well-designed Generative AI writing assistant can improve creativity and help authors generate ideas while still allowing human writers to make the final creative decisions.

---
#